In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

from beir.datasets.data_loader import GenericDataLoader
from beir import util

import os

import numpy as np

from tqdm.notebook import tqdm

/home/michele/.local/lib/python3.10/site-packages/beir/datasets/data_loader.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
import pathlib
from typing import Dict, List, Tuple


def data_preparation(dataset: str) -> List[str]:
    """
    Download the given dataset from beir and transform it in a list of concatenated titles and texts.

    Args:
        dataset (str): dataset name.

    Returns:
        List[str]: corpus: each string represent is a document title concatenated with document text.
    """
    
    #Download dataset and unzip the dataset
    url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
    out_dir = os.path.join(pathlib.Path(os.path.abspath('')), "datasets")
    data_path = util.download_and_unzip(url, out_dir)
    
    #Retrieve documents
    documents, _, _ = GenericDataLoader(data_path).load(split="test")
    
    return [document["title"]+" "+document["text"] for document in documents.values()]

In [3]:
corpus = data_preparation("scidocs")

  0%|          | 0/25657 [00:00<?, ?it/s]

In [4]:
corpus

['A hybrid of genetic algorithm and particle swarm optimization for recurrent network design An evolutionary recurrent network which automates the design of recurrent neural/fuzzy networks using a new evolutionary learning algorithm is proposed in this paper. This new evolutionary learning algorithm is based on a hybrid of genetic algorithm (GA) and particle swarm optimization (PSO), and is thus called HGAPSO. In HGAPSO, individuals in a new generation are created, not only by crossover and mutation operation as in GA, but also by PSO. The concept of elite strategy is adopted in HGAPSO, where the upper-half of the best-performing individuals in a population are regarded as elites. However, instead of being reproduced directly to the next generation, these elites are first enhanced. The group constituted by the elites is regarded as a swarm, and each elite corresponds to a particle within it. In this regard, the elites are enhanced by PSO, an operation which mimics the maturing phenomen

In [5]:
len(corpus)

25657

In [6]:
import spacy
    
class Cleaner:
    
    def __init__(self):
        #load the spacy model for lemmatization 
        self._nlp = spacy.load("en_core_web_lg", disable=['parser','ner'])

    def __call__(self, text: str ):
        return " ".join([token.lemma_ for token in self._nlp(text) if not token.is_stop and not token.is_punct])
    
cleaner=Cleaner()

In [7]:
cleaned_corpus=[]

for text in tqdm( corpus ):
    cleaned_corpus.append( cleaner(text) )

  0%|          | 0/25657 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [8]:
from multiprocessing import Pool

#load the spacy model for lemmatization 
_nlp = spacy.load("en_core_web_lg",disable=['parser','ner'])

def cleaner(text: str ):
    return " ".join([token.lemma_ for token in _nlp(text) if not token.is_stop and not token.is_punct])
    
with Pool() as p:
    cleaned_corpus=list(tqdm( p.imap(cleaner, corpus), 
                                            total=len(corpus),
                                            desc="documents cleaning"))

documents cleaning:   0%|          | 0/25657 [00:00<?, ?it/s]

In [9]:
vectorizer = TfidfVectorizer()

In [10]:
X = vectorizer.fit_transform(cleaned_corpus)

In [11]:
X

<25657x71051 sparse matrix of type '<class 'numpy.float64'>'
	with 1826805 stored elements in Compressed Sparse Row format>

In [ ]:
scores = X.dot( X.transpose() )

scores

(25657, 25657)

In [ ]:
def documents_pairs(scores: np.array, threshold: float):
    
    np.fill_diagonal(scores, 0)
    
    pairs = np.argwhere(scores > threshold)
    
    return pairs

In [ ]:
bho = documents_pairs(scores.toarray(), 0.85)
bho

In [ ]:
corpus[bho[0][0]]

'PD-L1 expression in human cancers and its association with clinical outcomes PD-L1 is an immunoinhibitory molecule that suppresses the activation of T cells, leading to the progression of tumors. Overexpression of PD-L1 in cancers such as gastric cancer, hepatocellular carcinoma, renal cell carcinoma, esophageal cancer, pancreatic cancer, ovarian cancer, and bladder cancer is associated with poor clinical outcomes. In contrast, PD-L1 expression correlates with better clinical outcomes in breast cancer and merkel cell carcinoma. The prognostic value of PD-L1 expression in lung cancer, colorectal cancer, and melanoma is controversial. Blocking antibodies that target PD-1 and PD-L1 have achieved remarkable response rates in cancer patients who have PD-L1-overexpressing tumors. However, using PD-L1 as an exclusive predictive biomarker for cancer immunotherapy is questionable due to the low accuracy of PD-L1 immunohistochemistry staining. Factors that affect the accuracy of PD-L1 immunohis

In [ ]:
corpus[bho[0][1]]

'Efficacy of PD-1 or PD-L1 inhibitors and PD-L1 expression status in cancer: meta-analysis OBJECTIVE To evaluate the relative efficacy of programmed cell death 1 (PD-1) or programmed cell death ligand 1 (PD-L1) inhibitors versus conventional drugs in patients with cancer that were PD-L1 positive and PD-L1 negative. DESIGN Meta-analysis of randomised controlled trials. DATA SOURCES PubMed, Embase, Cochrane database, and conference abstracts presented at the American Society of Clinical Oncology and European Society of Medical Oncology up to March 2018. REVIEW METHODS Studies of PD-1 or PD-L1 inhibitors (avelumab, atezolizumab, durvalumab, nivolumab, and pembrolizumab) that had available hazard ratios for death based on PD-L1 positivity or negativity were included. The threshold for PD-L1 positivity or negativity was that PD-L1 stained cell accounted for 1% of tumour cells, or tumour and immune cells, assayed by immunohistochemistry staining methods. RESULTS 4174 patients with advanced o